# Cyclistic Pandas Analysis

Full-dataset analysis using pandas.
Each step is timed and memory-measured via `StepTimer` from `utils/metrics.py`.

In [ ]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from utils.metrics import StepTimer

print("Imports OK")

## Step 1 — Load data

In [ ]:
with StepTimer("step1_load") as t:
    df = pd.read_csv("data/processed/trips_clean.csv")
print(f"Rows loaded: {len(df):,}")
df.head()

## Step 2 — Parse timestamps and compute ride length

In [ ]:
with StepTimer("step2_parse_timestamps") as t:
    df["started_at"] = pd.to_datetime(df["started_at"])
    df["ended_at"]   = pd.to_datetime(df["ended_at"])
    df["ride_length_minutes"] = (
        (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60
    )
df[["ride_id", "started_at", "ended_at", "ride_length_minutes"]].head()

## Step 3 — Quality checks and filtering

Look for missing values. Filter out negative or >24 hr rides.

In [ ]:
with StepTimer("step3_quality") as t:
    missing_values = df.isna().sum().sort_values(ascending=False)
    print("Missing values per column:\n", missing_values)

    print("\nRide length (minutes) summary:")
    print(df["ride_length_minutes"].describe())

    max_duration_minutes = 24 * 60
    mask_valid = (
        (df["ride_length_minutes"] > 0)
        & (df["ride_length_minutes"] <= max_duration_minutes)
    )
    print("\nRides before filtering:", len(df))
    print("Rides after filtering:",  mask_valid.sum())

    df_clean = df[mask_valid].copy()

## Step 4 — Member vs casual

Compare ride lengths and trip counts between members and casual riders.

In [ ]:
with StepTimer("step4_member_casual") as t:
    ride_length_summary = df_clean.groupby("member_casual")["ride_length_minutes"].describe()
    print("Ride length (minutes) by user type:\n", ride_length_summary)

    trip_share = df_clean["member_casual"].value_counts(normalize=True).rename("share")
    print("\nTrip share by user type:\n", (trip_share * 100).round(2).astype(str) + "%")

    df_clean["hour"]        = df_clean["started_at"].dt.hour
    df_clean["day_of_week"] = df_clean["started_at"].dt.day_name()

## Step 5 — Time and day patterns

Examine how rides vary by hour of day and day of week for members vs casual riders.

Members peak sharply at 8am and 5pm (commuter pattern); casuals peak in the afternoon and heavily on weekends (leisure/tourist pattern).

In [ ]:
with StepTimer("step5_time_patterns") as t:
    day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

    hourly_counts = df_clean.groupby(["hour","member_casual"]).size().unstack(fill_value=0)
    weekday_counts = (
        df_clean.groupby(["day_of_week","member_casual"]).size()
        .unstack(fill_value=0).reindex(day_order)
    )

# Hourly plot
plt.figure(figsize=(10, 6))
for user_type in hourly_counts.columns:
    plt.plot(hourly_counts.index, hourly_counts[user_type], marker="o", label=user_type)
plt.title("Trips by Hour of Day and User Type")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Trips")
plt.xticks(range(0, 24))
plt.legend(title="User type")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Weekday bar chart
bar_width = 0.35
x = range(len(day_order))
member_counts = weekday_counts["member"]
casual_counts = weekday_counts["casual"]

plt.figure(figsize=(10, 6))
plt.bar([i + bar_width/2 for i in x], casual_counts, width=bar_width, label="casual")
plt.bar([i - bar_width/2 for i in x], member_counts, width=bar_width, label="member")
plt.title("Trips by Day of Week and User Type")
plt.xlabel("Day of Week")
plt.ylabel("Number of Trips")
plt.xticks(list(x), day_order, rotation=45)
plt.legend(title="User type")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6 — Ride length distributions and bike type usage

In [ ]:
with StepTimer("step6_distributions") as t:
    max_minutes = 120
    bins = range(0, max_minutes + 5, 5)

    plt.figure(figsize=(10, 6))
    for user_type, color in [("casual","tab:red"),("member","tab:orange")]:
        subset = df_clean[df_clean["member_casual"] == user_type]
        plt.hist(
            subset["ride_length_minutes"].clip(upper=max_minutes),
            bins=bins, alpha=0.4, label=user_type, edgecolor="black",
        )
    plt.title("Ride Length Distribution (capped at 120 minutes)")
    plt.xlabel("Ride length (minutes)")
    plt.ylabel("Number of trips")
    plt.legend(title="User type")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    if "rideable_type" in df_clean.columns:
        bike_user_ct = pd.crosstab(df_clean["rideable_type"], df_clean["member_casual"])
        print("Bike type x user type (counts):\n", bike_user_ct)
        avg_dur_bike = (
            df_clean.groupby(["rideable_type","member_casual"])["ride_length_minutes"]
            .mean().round(1).unstack()
        )
        print("\nAvg ride length by bike type and user type:\n", avg_dur_bike)
    else:
        print("rideable_type not in dataset — bike-type analysis skipped.")

## Step 7 — Station-based insights

Heaviest step: group-by station + user type, average duration, across the full dataset.
This is the bottleneck step that Spark will parallelise.

In [ ]:
with StepTimer("step7_station_analysis") as t:
    min_trips = 30

    for user_type in ["member", "casual"]:
        top_start = (
            df_clean[df_clean["member_casual"] == user_type]["start_station_name"]
            .value_counts().head(10)
        )
        print(f"Top 10 start stations for {user_type}s:\n", top_start)

        top_end = (
            df_clean[df_clean["member_casual"] == user_type]["end_station_name"]
            .value_counts().head(10)
        )
        print(f"\nTop 10 end stations for {user_type}s:\n", top_end)

    avg_duration_station = (
        df_clean.groupby(["start_station_name","member_casual"])["ride_length_minutes"]
        .agg(["count","mean"])
        .reset_index()
    )
    avg_duration_station = avg_duration_station[avg_duration_station["count"] >= min_trips]

    print("\nAvg ride length (minutes) by start station and user type (>= 30 trips):")
    print(avg_duration_station.sort_values("mean", ascending=False).head(20))

## Step 8 — Time-of-day x day-of-week heatmaps

In [ ]:
with StepTimer("step8_heatmaps") as t:
    cat_type = pd.CategoricalDtype(categories=day_order, ordered=True)
    df_clean["day_of_week"] = df_clean["day_of_week"].astype(cat_type)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    for ax, user_type in zip(axes, ["member", "casual"]):
        subset = df_clean[df_clean["member_casual"] == user_type]
        pivot = subset.pivot_table(
            index="day_of_week", columns="hour",
            values="ride_id", aggfunc="count", fill_value=0,
        )
        sns.heatmap(pivot, ax=ax, cmap="Blues", cbar_kws={"label":"Number of trips"})
        ax.set_title(f"Trips Heatmap: {user_type}")
        ax.set_xlabel("Hour of day")
        ax.set_ylabel("Day of week")
    plt.tight_layout()
    plt.show()

## Step 9 — Segment comparisons for business insights

Weekend vs weekday, commute vs leisure, ride length bands.

In [ ]:
with StepTimer("step9_segments") as t:
    df_clean["is_weekend"] = df_clean["day_of_week"].isin(["Saturday","Sunday"])

    commute_hours = list(range(6,10)) + list(range(16,20))
    df_clean["is_commute_hour"] = df_clean["hour"].isin(commute_hours)

    bins_dur = [0, 15, 45, float("inf")]
    labels_dur = ["short (<=15m)", "medium (15-45m)", "long (>45m)"]
    df_clean["duration_band"] = pd.cut(
        df_clean["ride_length_minutes"], bins=bins_dur, labels=labels_dur, right=True
    )

    weekend_summary = (
        df_clean.groupby(["member_casual","is_weekend"])["ride_id"]
        .count().unstack(fill_value=0)
    )
    print("Weekend vs weekday trips by user type:\n", weekend_summary)

    commute_summary = (
        df_clean.groupby(["member_casual","is_commute_hour"])["ride_id"]
        .count().unstack(fill_value=0)
    )
    print("\nCommute-hour vs other-hour trips by user type:\n", commute_summary)

    band_summary = pd.crosstab(
        df_clean["member_casual"], df_clean["duration_band"], normalize="index"
    ) * 100
    print("\nRide length bands (% of trips) by user type:\n", band_summary.round(1))